# Accessing Experiment Data From S3

This notebook walks through how to access the S3 experiment data in HDF5 format using lazy loaders so that the whole data is not loaded into the RAM.

To view your data, edit the `S3Backend` parameters with your `bucket`, `endpoint_url`, and `prefix` values.

## Imports

The notebook uses `pd_xray` as backend to perform all the operations. This makes the experience a bit more seamless.

For this notebook, we will be using `S3Backend` for the backend and `HDF5Reader` as the file reader. They work together to access the HDF5 files in a lazy manner.

Apart from those, `ImageProcessor` can be utilised to process the frames.

In [ ]:
from pd_xray.data import S3Backend, HDF5Reader
from pd_xray.visualisation import view_frames, apply_and_view, save_selection
from pd_xray.processing import ImageProcessor

## Finding the correct campaign and prefix

If you don't already know the correct campaign you are looking for, and the prefix; you can start from the base bucket and display the files.

Enter the correct bucket name and don't provide a prefix. Then list the files. This will give you an idea about the main paths under your bucket.

Then walk file by file (i.e., keep adding into prefix: `prefix="campaign"`, then `prefix="campaign/data"` etc.) until you find what you are looking for.

In [ ]:
with S3Backend(
    bucket="my-bucket",
    endpoint_url="https://end.point.to.s3.ac.uk",
) as backend:
    backend.connect()
    files = backend.list_files()
    for file in files:
        print(file)

### Listing experiment files

Once you find the correct prefix; you can list all the files under that prefix. This will give you information about the online file

In [ ]:
with S3Backend(
    bucket="my-bucket",
    endpoint_url="https://end.point.to.s3.ac.uk",
    prefix="campaign_1/data/my_data"
) as backend:
    backend.connect()
    files = backend.list_files()
    for i, file in enumerate(files):
        if i >= 50:
            print(f"{i}: {file}")

## Accessing and visualising the data

After finding the exact file you are looking for; you can read and visualise that file.

Increase the `block_size` to read through files faster. Note that, even though this will result if fewer look-ups to reach your data, it may be too much for your bandwith. If higher `block_size` crashes the kernel, reduce it.

```python
block_size = 64  * 1028 * 1028   # 64 MB
block_size = 256 * 1028 * 1028  # 256 MB
```

Initialise the `HDF5Reader` along with the backend.

In [ ]:
backend = S3Backend(
    bucket="my-bucket",
    endpoint_url="https://end.point.to.s3.ac.uk",
    prefix="campaign_1/data/my_data",
    block_size=256 * 1024 * 1024,  # 128 MB
)
backend.connect()
reader = HDF5Reader()

### Read the data

Use the file name of your experiment HDF5 file and lazy read it.

Once you establish the connection, you can read sections of the data to read a subsection (this is recommended as most of the experiment data is too large and won't fit into normal laptop RAM, i.e. >50 GB)

In [ ]:
with reader.lazy_open_remote(backend, "my_experiment.h5") as arr:
    print(f"Data shape: {arr.shape}")
    print(f"Data dtype: {arr.dtype}")
    section = arr[0:10, :, :]
    print(f"Section shape: {section.shape}")

### Visualise

Now that you have a section of the data, you can visualise that and apply image processing with `ImageProcessor()`

In [ ]:
view_frames(section, title="First 10 frames")

In [ ]:
proc = (
    ImageProcessor()
    .gaussian_blur(sigma=1.5)
    .normalise()
)
processed = apply_and_view(section, proc)

## Download file to your local

In [ ]:
backend.read_file_to_local(
    remote_path="your_h5_file.h5",
    local_path="/local/download/path",
)